<a href="https://colab.research.google.com/github/nour-houda-melkii/Esprit--PIDS-4DS10-2026-DataVerse/blob/Geopolitical_Agent/1_Data_Scraping.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================================
# NOTEBOOK 1: DATA SCRAPING (Historical 2018-2025)
# ============================================================================
# Runtime: 1-3 hours
# Output: geopolitical_historical_2018_2025_RAW.csv
# ============================================================================

print("="*80)
print("🚀 GEOPOLITICAL FACTOR: DATA SCRAPING")
print("="*80)
print("\nThis will scrape historical events from 2018-2025")
print("Estimated time: 1-3 hours")
print("="*80 + "\n")

# ----------------------------------------------------------------------------
# CELL 1: Install Dependencies
# ----------------------------------------------------------------------------

!pip install -q pandas numpy requests tqdm

print("✅ Dependencies installed!")

# ----------------------------------------------------------------------------
# CELL 2: Import Libraries
# ----------------------------------------------------------------------------

import requests
import pandas as pd
from datetime import datetime, timedelta
import time
import os
from typing import List, Dict, Tuple
import json
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

print("✅ Libraries imported!")

# ----------------------------------------------------------------------------
# CELL 3: Define Scraper Classes
# ----------------------------------------------------------------------------

class GDELTScraper:
    """Scrapes news events from GDELT."""

    BASE_URL = "http://api.gdeltproject.org/api/v2/doc/doc"

    CURRENCY_QUERIES = {
        'USD': 'United States OR USA OR dollar OR "Federal Reserve" OR America',
        'EUR': 'Europe OR eurozone OR Germany OR France OR ECB OR "European Union"',
        'JPY': 'Japan OR yen OR "Bank of Japan" OR Tokyo',
        'GBP': 'Britain OR UK OR pound OR "Bank of England" OR London',
        'CHF': 'Switzerland OR Swiss OR franc OR "Swiss National Bank"'
    }

    EVENT_KEYWORDS = (
        'war OR conflict OR invasion OR attack OR military OR sanctions OR '
        'election OR crisis OR disaster OR trade OR tariff OR '
        'terrorism OR nuclear OR protest OR coup'
    )

    def __init__(self, start_year, end_year):
        self.start_year = start_year
        self.end_year = end_year
        self.all_data = []

    def scrape_all(self):
        """Scrape all GDELT data."""
        months = self._generate_months()

        print(f"\n📡 Scraping GDELT: {len(months)} months to process...")

        for start_date, end_date, month_id in tqdm(months, desc="GDELT Progress"):

            month_data = []

            for currency in self.CURRENCY_QUERIES.keys():
                df = self._scrape_month(start_date, end_date, currency)
                if len(df) > 0:
                    month_data.append(df)
                time.sleep(1)  # Rate limiting

            if month_data:
                month_df = pd.concat(month_data, ignore_index=True)
                self.all_data.append(month_df)
                print(f"   ✓ {month_id}: {len(month_df)} events")

            time.sleep(2)

        if self.all_data:
            return pd.concat(self.all_data, ignore_index=True)
        return pd.DataFrame()

    def _generate_months(self):
        """Generate month ranges."""
        months = []
        for year in range(self.start_year, self.end_year + 1):
            for month in range(1, 13):
                if year == datetime.now().year and month > datetime.now().month:
                    break

                start = datetime(year, month, 1)
                if month == 12:
                    end = datetime(year, 12, 31)
                else:
                    end = datetime(year, month + 1, 1) - timedelta(days=1)

                months.append((
                    start.strftime('%Y%m%d'),
                    end.strftime('%Y%m%d'),
                    f"{year}-{month:02d}"
                ))
        return months

    def _scrape_month(self, start_date, end_date, currency):
        """Scrape one month of GDELT data."""
        query = f"({self.CURRENCY_QUERIES[currency]}) AND ({self.EVENT_KEYWORDS})"

        params = {
            'query': query,
            'mode': 'ArtList',
            'maxrecords': 250,
            'format': 'json',
            'startdatetime': start_date + '000000',
            'enddatetime': end_date + '235959',
            'sourcelang': 'eng'
        }

        try:
            response = requests.get(self.BASE_URL, params=params, timeout=60)
            response.raise_for_status()
            data = response.json()

            if 'articles' not in data:
                return pd.DataFrame()

            events = []
            for article in data['articles']:
                events.append({
                    'date': article.get('seendate', '')[:8],
                    'time': article.get('seendate', '')[8:],
                    'currency_primary': currency,
                    'title': article.get('title', ''),
                    'url': article.get('url', ''),
                    'source': f"GDELT-{article.get('domain', 'unknown')}"
                })

            return pd.DataFrame(events)

        except Exception as e:
            return pd.DataFrame()

print("✅ Scraper class defined!")

# ----------------------------------------------------------------------------
# CELL 4: Run Scraper
# ----------------------------------------------------------------------------

print("\n" + "="*80)
print("🚀 STARTING DATA COLLECTION")
print("="*80)
print("\n⏰ This will take 1-3 hours - Colab will stay running")
print("⚠️  DO NOT CLOSE THIS TAB!\n")

# Initialize scraper
scraper = GDELTScraper(
    start_year=2018,
    end_year=2025
)

# Run scraping
df = scraper.scrape_all()

print("\n" + "="*80)
print("✅ SCRAPING COMPLETE!")
print("="*80)
print(f"Total events collected: {len(df):,}")
print(f"Date range: {df['date'].min()} to {df['date'].max()}")
print(f"Currencies: {df['currency_primary'].nunique()}")
print("="*80 + "\n")

# ----------------------------------------------------------------------------
# CELL 5: Save Data
# ----------------------------------------------------------------------------

# Save to CSV
output_file = 'geopolitical_historical_2018_2025_RAW.csv'
df.to_csv(output_file, index=False, encoding='utf-8-sig')

print(f"💾 Saved to: {output_file}")
print(f"📊 File size: {os.path.getsize(output_file) / 1024 / 1024:.2f} MB")

# Download to your computer
from google.colab import files
files.download(output_file)

print("\n✅ File downloaded to your computer!")
print("📋 Sample of data:")
print(df.head(10))

# ----------------------------------------------------------------------------
# CELL 6: Data Statistics
# ----------------------------------------------------------------------------

print("\n" + "="*80)
print("📊 DATA STATISTICS")
print("="*80)

print(f"\nTotal Events: {len(df):,}")
print(f"Date Range: {df['date'].min()} to {df['date'].max()}")
print(f"\nBy Currency:")
print(df['currency_primary'].value_counts())
print(f"\nTop 10 Sources:")
print(df['source'].value_counts().head(10))

print("\n✅ NOTEBOOK 1 COMPLETE!")
print("➡️  Next: Run Notebook 2 (Data Cleaning)")

🚀 GEOPOLITICAL FACTOR: DATA SCRAPING

This will scrape historical events from 2018-2025
Estimated time: 1-3 hours

✅ Dependencies installed!
✅ Libraries imported!
✅ Scraper class defined!

🚀 STARTING DATA COLLECTION

⏰ This will take 1-3 hours - Colab will stay running
⚠️  DO NOT CLOSE THIS TAB!


📡 Scraping GDELT: 96 months to process...


GDELT Progress:   0%|          | 0/96 [00:00<?, ?it/s]

   ✓ 2018-01: 250 events


GDELT Progress:   1%|          | 1/96 [04:16<6:46:48, 256.93s/it]

   ✓ 2018-02: 1250 events


...

GDELT Progress: 100%|██████████| 96/96 [4:08:37<00:00, 155.39s/it]



✅ SCRAPING COMPLETE!
Total events collected: 44,750
Date range: 20180101 to 20260101
Currencies: 5

💾 Saved to: geopolitical_historical_2018_2025_RAW.csv
📊 File size: 9.35 MB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


✅ File downloaded to your computer!
📋 Sample of data:
       date      time currency_primary  \
0  20180109  T094500Z              CHF   
1  20180109  T081500Z              CHF   
2  20180108  T133000Z              CHF   
3  20180108  T101500Z              CHF   
4  20180105  T150000Z              CHF   
5  20180109  T120000Z              CHF   
6  20180108  T160000Z              CHF   
7  20180104  T164500Z              CHF   
8  20180104  T180000Z              CHF   
9  20180109  T174500Z              CHF   

                                               title  \
0  $55 Billion Profit for Giant Asset Manager the...   
1  Swiss National Bank Says It Made CHF54 Billion...   
2  Suisse : Larmée veut mobiliser ses soldats par...   
3  Suisse : Larmée veut mobiliser ses soldats par...   
4  Politique monétaire : Un bénéfice de 50 millia...   
5  Swiss National Bank : Switzerland Central Bank...   
6  Pound to Swiss Franc Exchange Rate Could Shed ...   
7  Travail : lUnion syndicale suis